<a target="_blank" href="https://colab.research.google.com/github/TUIlmenauAMS/Videocoding/blob/main/LecturesJupterNotebooks/Lecture9/Lecture9.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# <font color="#B85C38">Lecture 9: Transforms 2</font>

*Video Coding · Prof. Gerald Schuller*

**Integer Transform of H.264/AVC**

In previous standards, the DCT was defined as the ideal transform, with **unlimited accuracy**. This has the problem, that we have encoders with **limited accuracy** for the DCT, and decoders, with different accuracy for the inverse DCT. This then leads to **reconstruction errors** from the different implementations. Also, if we want to implement the DCT with low error, we need an arithmetic with long word length and floating point implementation, which makes the implementation complex. This is a problem for low end processors, it makes the coder hardware more expensive. Since Video decoding plays an increasing role in consumer hardware, where cost is very important, this problem was addressed in H.264. They solved the problem by specifying the **DCT with integer arithmetic**, such that the rounding errors we make are now known (since it is specified in the standard). In this way, we can still obtain **perfect reconstruction** (in the absence of quantization), because in the standard we can specify the **inverse** DCT with **integer arithmetic**, such that we obtain the exact inverse. In this way, the rounding errors „fit“ to each other, to obtain perfect reconstruction. In this way we get a different transform, which is not quite the DCT, but only similar to it.

The approach was to take the DCT matrix, **multiply** it with a **certain factor**, and **round** it to obtain a transform matrix with integer entries. A factor which turned out to be suitable is $\alpha=2.5$ (see also: H. Malvar et. Al: „Low Complexity Transform and Quantization in H.264/AVC“, IEEE Trans. on Circuits and Systems for Video Technology, July 2003, to find under: **ieeexplore.ieee.org** inside the University Net). It still leads to a good coding gain, but allows a low wordlength processing (16 bit arithmetic).

H.264 uses 8x8 and also 4x4 transform matrices. Here we look at the 4x4 version. We start with the usual 4x4 DCT type 2 (shown are 4 digits after the decimal point),

$$\boldsymbol T= \left[\matrix { 0.5000 &   0.5000   & 0.5000   & 0.5000 \\
0.6533&0.2706&-0.2706&-0.6533 \\
0.5000&-0.5000& -0.5000&0.5000 \\
0.2706&-0.6533&0.6533&-0.2706
} \right]$$

Here we assume that we multiply our signal vector **x** from the right hand side, to obtain the transformed signal: $\boldsymbol y^T= \boldsymbol T.\boldsymbol x^T$ , and for the inverse: $\boldsymbol x^T=T^{-1}.\boldsymbol y^T$.
With the factor of $\alpha=2.5$ we get

$$
2.5 \cdot \boldsymbol T  = \left[ \matrix{
1.2500  &  1.2500 &   1.2500 &   1.2500 \\
1.6332  &  0.6765 &  -0.6765 &  -1.6332 \\
1.2500  & -1.2500 &  -1.2500 &   1.2500 \\
0.6765  &  -1.6332 &   1.6332 &  -0.6765} \right]
$$

Now we can round it to obtain a useful transform matrix with integer entries,

$$\boldsymbol H := round(2.5 \cdot \boldsymbol T)= \left[ \matrix{
1  &   1  &   1  &   1\\
2  &   1  &  -1  &  -2\\
1  &  -1  &  -1   &  1\\
1  &  -2  &  2   & -1}\right]
$$

This is the **forward transform matrix used by H.264**. We see that it is quite similar to the WHT, but gives better compression performance. Malvar writes that for a stationary Gauss-Markov input with correlation coefficient $\rho = 0.9$ the coding gain for the DCT is 5.39 dB, whereas for this **integer** version it is 5.38 dB.

Generation of this **stationary Gauss-Markov input** y(n):

![Lecture9-1.PNG](https://github.com/TUIlmenauAMS/Videocoding/blob/main/LecturesJupterNotebooks/Lecture9/Img-Lecture9/Lecture9-1.PNG?raw=1)

or: $y( n )=x( n )+ 0.9 \cdot y( n-1 )$, with

x(n): random memoryless gaussian input

y(n): filtered output, the stationary Gauss-Markov signal. In the z-domain we obtain a pole at z=0.9 in the transfer function, which makes it a low pass. This **fits to natural images**, because they also have a low pass characteristic. But also observe that this is only a very crude and simple approximation of a natural image, but at least this gives us a simple model of a natural image.

The **coding gain** is defined as the **arithmetic average divided by the geometric average** of the squares of the subband values, $y^{2}_k$, to give us an estimate of the compression performance of a given subband decomposition (in dB). The arithmetic average is our usual average as the sum of values divided by the number of values, the geometric average is using the product and the Nth root instead of the sum and the division by N:

$$
\text{coding-gain}:\frac{\frac{1} {N} \cdot \sum _{k=0} ^ {N-1} y^2_k} {{\sqrt [N]{ \prod _{k=0}^ {N-1} y^2_k}}}
$$

See also: Jayant, Noll: “Digital Coding of Waveforms”, Prentice Hall.

The **arithmetic average** is the energy or power of the signal. **Parsevals Theorem** says that the sum of the  power of the subband signals is identical to the signals power in the time or space domain. If we take the log2 of the arithmetic average above, we get an estimate of the needed **bits per sample in the time or space domain** (times 2 since we have the square), also called the Entropy.

If we take the log2 of the **geometric average**, we get an estimate of the number of needed **bits per sample** to encode the signal in the **subbands**.

$$
\log_2(\sqrt [N] {\prod _{k=0} ^ {N-1} y^2_k})=\sum_{k=0}^{N-1}  \log_2(y^2_k)/N
$$

Here, $\log_2(y^2_k)$ is an estimated number of bits (times 2 because of the square) per sample in subband k, hence the sum is the estimate of the average number of bits per sample over all subbands (Entropy over the subbands).

So if we take the $log_2$ of the coding gain as defined above, we obtain the difference in needed bits (times 2) for the direct and the subband coding. If we take the 10*log10, we obtain the number in dB, where we have about **6dB per bit**. The usual is the **coding gain in dB**. We would like to have the coding gain as high as possible.

**Observe:** If the coding gain is 1, or 0 dB, all the subband values $y_k$ are identical. Only in this case, the geometric and arithmetic average become identical. The more different the $y_k$ become, the higher the coding gain, with the extreme of a $y_k$ being 0. In this case we have an infinite coding gain. If we have **6dB coding gain, we save 1 bit/sample**. See the Book by Jayant/Noll.

So at least for this artificial signal it is only a very small loss in coding gain of only 0.01 dB for the rounded DCT (or a cost of only 0.01/6 bits/sample) . For a WHT the loss in coding gain would be clearly higher. Observe that we can use the same approach (with the same factor) to obtain an integer 8x8 transform.

We still need an **integer** value **inverse**, for the decoder. Is it possible to get an integer valued **exact** inverse?

We can simply first compute the inverse of **H**,

$$
\boldsymbol H ^{-1}= \left[ \matrix{
0.2500  &  0.2000  &  0.2500 &   0.1000\\
0.2500  &  0.1000 &  -0.2500  & -0.2000\\
0.2500  & -0.1000 &  -0.2500 &   0.2000 \\
0.2500 &  -0.2000 &   0.2500 &  -0.1000}\right]
$$

Here we could use again a factor to obtain integer values for the inverse transform. In this way we would obtain the original, but scaled with the two factors. But the goal is to have **factors as small as possible** to have lower **wordlengths** for the integer arithmetic. The trick used here is to apply **different factors** to the columns, and allow factors which can be implemented with a shift operation (e.g. 0.5). These factors are 4;5;4;5.

With these factors we get the matrix

$$
\boldsymbol {\tilde H}_{inv} =\left[ \matrix{
1 &1& 1& 1/2\\
1 &1/2& -1&-1\\
1 & -1/2&-1& 1\\
1 & -1& 1& -1/2}\right]
$$

This means we get the inverse as

$$\boldsymbol H ^{-1}  = \boldsymbol {\tilde H}_{inv} \cdot  diag(1/4,1/5,1/4,1/5) $$
This shows that we just extracted the diagonal matrix from the inverse. The diagonal matrix does not need to be computed explicitly, because we can factor it into the inverse quantization of the decoder.

So, in this way we obtained the exact inverse with very simple numbers or fractions, easy to implement!

A fast implementation:

![Lecture9-2.PNG](https://github.com/TUIlmenauAMS/Videocoding/blob/main/LecturesJupterNotebooks/Lecture9/Img-Lecture9/Lecture9-2.png?raw=1)

(From [1]).

**Dynamic range of values after the transform:**

(important for the fixed wordlength integer arithmetic). Assume we have an input signal with a maximum value of A, for instance an image with  brightness levels A (for the worst case this would be the maximum value). Then we have a signal vector containing the values +-A (for instance for the chrominance values, which can also be negative), which is here multiplied from the right hand side. If we take the transform matrix **H** from last time, and if one column of **x** has for instance is the vector $[A,A,-A,-A]^T$ (as a worst case again) then the multiplication with the second row of H results to **6A**.  This would also be the maximum value for this matrix. If we then also transform the rows of our image, we get a maximum value of $6*6A= 36A$. This means that the dynamic range we have for our subband coefficients increases by $\log_2(36)=5.17$ bits compared to the dynamic range of the original images (we would need 5.17 bits or rather 6 bits more for our fixed wordlength integer arithmetic). This is an overhead which we need to provide in our coding signal processor. This is also the reason why we wanted to have our factor $\alpha$ as small as possible.

For the inverse matrix, in the decoder, the factor is somewhat smaller. Here we get a factor of 4, leading to a factor of 16 for rows and colums, and hence 4 additional bits for the dynamic range for the decoded subband samples. Observe that this also means a reduced (maximum) information content in the subband signals, which is the result of the quantization in the encoder.

These effects become important if we want to implement our algorithm with integer arithmetic, with limited word length. H.264 is made such that it can be implemented with **16 bit** arithemtic (in encoder and decoder, and which is sufficient, as we see here), which enables the implementation into cheap hardware.

## <font color="#B85C38">Literature:</font>

- Jayant, Noll: “Digital Coding of Waveforms”, Prentice Hall.

- H. Malvar et. Al: „Low Complexity Transform and Quantization in H.264/AVC“, IEEE Transactions on Circuits and Systems for Video Technology, July 2003

- Kalva, H.: "The H.264 Video Coding Standard "  
  IEEE  Transactions on Multimedia,  
  Volume: 13 , Issue: 4  
  Digital Object Identifier: 10.1109/MMUL.2006.93  
  Publication Year: 2006 , Page(s): 86 - 90

- Ugur, K.; Andersson, K.; Fuldseth, A.; Bjøntegaard, G.; Endresen, L.P.; Lainema, J.; Hallapuro, A.; Ridge, J.; Rusanovskyy, D.; Cixun Zhang; Norkin, A.; Priddle, C.; Rusert, T.; Samuelsson, J.; Sjöberg, R.; Zhuangfei Wu:  
  "Low complexity video coding and the emerging HEVC standard"  
  Picture Coding Symposium (PCS), 2010  
  Digital Object Identifier: 10.1109/PCS.2010.5702540  
  Publication Year: 2010 , Page(s): 474 - 477

- Ugur, K.; Andersson, K.; Fuldseth, A.; Bjontegaard, G.; Endresen, L.P.; Lainema, J.; Hallapuro, A.; Ridge, J.; Rusanovskyy, D.; Cixun Zhang; Norkin, A.; Priddle, C.; Rusert, T.; Samuelsson, J.; Sjoberg, R.; Zhuangfei Wu:  
  "High Performance, Low Complexity Video Coding and the Emerging HEVC Standard ",  
  IEEE Transactions on Circuits and Systems for Video Technology,  
  Volume: 20 , Issue: 12  
  Digital Object Identifier: 10.1109/TCSVT.2010.2092613  
  Publication Year: 2010 , Page(s): 1688 - 1697

- Marpe, D.; Schwarz, H.; Bosse, S.; Bross, B.; Helle, P.; Hinz, T.; Kirchhoffer, H.; Lakshman, H.; Nguyen, T.; Oudin, S.; Siekmann, M.; Suhring, K.; Winken, M.; Wiegand, T.:  
  "Video Compression Using Nested Quadtree Structures, Leaf Merging, and Improved Techniques for Motion Representation and Entropy Coding "  
  IEEE Transactions on Circuits and Systems for Video Technology,  
  Volume: 20 , Issue: 12  
  Digital Object Identifier: 10.1109/TCSVT.2010.2092615  
  Publication Year: 2010 , Page(s): 1676 - 1687

(All to be found at **ieeexplore.ieee.org** from inside the TU Ilmenau Network)